<a href="https://colab.research.google.com/github/Rais-Ataullov/BigData/blob/lab1/L1_Apache_Spark_Tasks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Решите следующие задачи для данных велопарковок Сан-Франциско (trips.csv, stations.csv):

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
import pyspark.sql as sql

conf = SparkConf().setAppName("L1_Apache_Spark_Tasks").setMaster('local[*]')

sc = SparkContext(conf=conf)
spark = SparkSession(sc)

In [2]:
tripData = spark.read\
.option("header", True)\
.option("inferSchema", True)\
.option("timestampFormat", 'M/d/y H:m')\
.csv("trips.csv").dropna()

stationData = spark.read\
.option("header", True)\
.option("inferSchema", True)\
.option("timestampFormat", 'M/d/y')\
.csv("stations.csv").dropna()

In [3]:
tripData.show(10)

+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|  id|duration|         start_date|  start_station_name|start_station_id|           end_date|    end_station_name|end_station_id|bike_id|subscription_type|zip_code|
+----+--------+-------------------+--------------------+----------------+-------------------+--------------------+--------------+-------+-----------------+--------+
|4130|      71|2013-08-29 10:16:00|Mountain View Cit...|              27|2013-08-29 10:17:00|Mountain View Cit...|            27|     48|       Subscriber|   97214|
|4251|      77|2013-08-29 11:29:00|  San Jose City Hall|              10|2013-08-29 11:30:00|  San Jose City Hall|            10|     26|       Subscriber|   95060|
|4299|      83|2013-08-29 12:02:00|South Van Ness at...|              66|2013-08-29 12:04:00|      Market at 10th|            67|    319|       Subscriber|   94103|
|4927|    

In [4]:
stationData.show(10)

+---+--------------------+------------------+-------------------+----------+--------+-------------------+
| id|                name|               lat|               long|dock_count|    city|  installation_date|
+---+--------------------+------------------+-------------------+----------+--------+-------------------+
|  2|San Jose Diridon ...|         37.329732|-121.90178200000001|        27|San Jose|2013-08-06 00:00:00|
|  3|San Jose Civic Ce...|         37.330698|        -121.888979|        15|San Jose|2013-08-05 00:00:00|
|  4|Santa Clara at Al...|         37.333988|        -121.894902|        11|San Jose|2013-08-06 00:00:00|
|  5|    Adobe on Almaden|         37.331415|          -121.8932|        19|San Jose|2013-08-05 00:00:00|
|  6|    San Pedro Square|37.336721000000004|        -121.894074|        15|San Jose|2013-08-07 00:00:00|
|  7|Paseo de San Antonio|         37.333798|-121.88694299999999|        15|San Jose|2013-08-07 00:00:00|
|  8| San Salvador at 1st|         37.330165|-

In [5]:
stationData.createOrReplaceTempView("stations")
tripData.createOrReplaceTempView("trips")

1. Найти велосипед с максимальным временем пробега.

In [6]:
total_durationData = spark.sql("""
SELECT trips.bike_id, SUM(duration) as total_duration FROM trips GROUP BY bike_id ORDER BY total_duration DESC;
""")

total_durationData.show(1)

+-------+--------------+
|bike_id|total_duration|
+-------+--------------+
|    593|        939949|
+-------+--------------+
only showing top 1 row


2. Найти наибольшее геодезическое расстояние между станциями.

In [7]:
max_distanceData = spark.sql("""
WITH station_pairs AS (
    SELECT
        s1.id AS station1_id,
        s1.name AS station1_name,
        s1.lat AS lat1,
        s1.long AS lon1,
        s2.id AS station2_id,
        s2.name AS station2_name,
        s2.lat AS lat2,
        s2.long AS lon2
    FROM stations s1
    CROSS JOIN stations s2
    WHERE s1.id < s2.id
),

station_distances AS (
    SELECT
        station1_id,
        station1_name,
        station2_id,
        station2_name,
        -- Радиус Земли в км = 6371
        6371 * 2 * ASIN(
            SQRT(
                POWER(SIN(RADIANS(lat2 - lat1) / 2), 2) +
                COS(RADIANS(lat1)) * COS(RADIANS(lat2)) *
                POWER(SIN(RADIANS(lon2 - lon1) / 2), 2)
            )
        ) AS distance_km
    FROM station_pairs
)

SELECT
    station1_name,
    station2_name,
    ROUND(distance_km, 2) AS distance_km
FROM station_distances
WHERE distance_km IS NOT NULL
ORDER BY distance_km DESC
LIMIT 1;
""")

max_distanceData.show(truncate=False)

+--------------------------+----------------------+-----------+
|station1_name             |station2_name         |distance_km|
+--------------------------+----------------------+-----------+
|SJSU - San Salvador at 9th|Embarcadero at Sansome|69.92      |
+--------------------------+----------------------+-----------+



3. Найти путь велосипеда с максимальным временем пробега через станции.

In [12]:
total_durationData.createOrReplaceTempView("total_duration")

max_duration_bike_way = spark.sql("""
WITH bike_total_duration AS (
    SELECT bike_id, SUM(duration) AS total_duration
    FROM trips
    GROUP BY bike_id
),
max_bike AS (
    SELECT bike_id
    FROM bike_total_duration
    ORDER BY total_duration DESC
    LIMIT 1
),
bike_trips AS (
    SELECT
        start_date,
        end_date,
        start_station_name,
        end_station_name,
        ROW_NUMBER() OVER (ORDER BY start_date) AS trip_number
    FROM trips
    WHERE bike_id = (SELECT bike_id FROM max_bike)
),
all_stops AS (
    SELECT start_date AS time, start_station_name AS station_name
    FROM bike_trips

    UNION ALL

    SELECT end_date AS time, end_station_name AS station_name
    FROM bike_trips
),
ordered_stops AS (
    SELECT
        time,
        station_name,
        LAG(station_name) OVER (ORDER BY time) AS prev_station
    FROM all_stops
)
SELECT
    ROW_NUMBER() OVER (ORDER BY time) AS stop_order,
    station_name,
    time
FROM ordered_stops
WHERE station_name != prev_station OR prev_station IS NULL
ORDER BY time;
""")

max_duration_bike_way.show(truncate=False)

+----------+---------------------------------------------+-------------------+
|stop_order|station_name                                 |time               |
+----------+---------------------------------------------+-------------------+
|1         |Post at Kearney                              |2013-08-29 12:00:00|
|2         |2nd at South Park                            |2013-08-29 12:08:00|
|3         |San Francisco Caltrain (Townsend at 4th)     |2013-08-31 09:07:00|
|4         |Civic Center BART (7th at Market)            |2013-08-31 09:25:00|
|5         |South Van Ness at Market                     |2013-08-31 16:11:00|
|6         |Powell Street BART                           |2013-08-31 16:19:00|
|7         |Grant Avenue at Columbus Avenue              |2013-08-31 17:55:00|
|8         |Post at Kearney                              |2013-08-31 18:50:00|
|9         |South Van Ness at Market                     |2013-08-31 19:03:00|
|10        |Market at 4th                           

4. Найти количество велосипедов в системе.

In [ ]:
bikes_count = spark.sql("""
SELECT COUNT(DISTINCT bike_id) AS bike_count FROM trips;
""")

bikes_count.show()

+----------+
|bike_count|
+----------+
|       700|
+----------+



5. Найти пользователей потративших на поездки более 3 часов (10800 сек).

In [ ]:
users_with_trips_more_3_hours = spark.sql("""
SELECT
    zip_code,
    subscription_type,
    SUM(duration) AS total_duration_seconds
FROM trips
GROUP BY zip_code, subscription_type
HAVING SUM(duration) > 3 * 3600
""")

users_with_trips_more_3_hours.show()

+--------+-----------------+----------------------+
|zip_code|subscription_type|total_duration_seconds|
+--------+-----------------+----------------------+
|   95125|       Subscriber|                590806|
|   95008|         Customer|                758377|
|    2465|         Customer|                 18241|
|   94619|       Subscriber|                404485|
|   94502|         Customer|                 62657|
|   92869|         Customer|                 12270|
|   49518|         Customer|                 33839|
|   29464|         Customer|                104615|
|   80220|         Customer|                 94992|
|      55|         Customer|                853637|
|    3141|         Customer|                 14423|
|   30307|         Customer|                 25457|
|   94035|         Customer|                165289|
|   94010|       Subscriber|               3539415|
|   90210|         Customer|                690278|
|   19444|         Customer|                 14868|
|   95118|  

In [ ]:
sc.stop()